# GPU Verification

Verify that Google Colab T4 GPU is available and properly configured for training.

In [1]:
import torch
import sys

print("=" * 60)
print("GPU VERIFICATION")
print("=" * 60)

# Check PyTorch version
print(f"PyTorch Version: {torch.__version__}")
print(f"Python Version: {sys.version}")

# Check CUDA availability
print(f"\nCUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"Current GPU: {torch.cuda.current_device()}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    
    # Get GPU memory info
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Memory: {gpu_memory:.2f} GB")
    
    # Test GPU with a simple tensor operation
    test_tensor = torch.randn(1000, 1000).cuda()
    print(f"\n✅ GPU Test Successful! Tensor device: {test_tensor.device}")
else:
    print("\n❌ WARNING: GPU not available! Training will be slow on CPU.")
    print("Make sure you selected 'GPU' runtime in Colab.")

print("=" * 60)

GPU VERIFICATION
PyTorch Version: 2.9.0+cu126
Python Version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]

CUDA Available: True
CUDA Version: 12.6
Number of GPUs: 1
Current GPU: 0
GPU Name: Tesla T4
GPU Memory: 15.83 GB

✅ GPU Test Successful! Tensor device: cuda:0


# ResNet101 - KHOTAA Diabetic Foot Ulcer Classification

## 🚀 Google Colab Setup Instructions

**Before running this notebook:**

1. **Select GPU Runtime:**
   - Click `Runtime` → `Change runtime type`
   - Hardware accelerator: **GPU** (T4)
   - Click **Save**

2. **Upload Dataset to Google Drive:**
   - Go to [drive.google.com](https://drive.google.com)
   - Upload the entire `dfu-dataset-annotated-into-4-classes` folder
   - Remember the location (e.g., `MyDrive/dfu-dataset-annotated-into-4-classes`)

3. **Run Cells in Order:**
   - Start with GPU Verification (Cell 2)
   - Clone repository (Cell 6)
   - Mount Google Drive and set dataset path (Cell 10)
   - Continue with remaining cells

**Estimated Training Time:** ~2-3 hours on T4 GPU for 5-fold cross-validation

## 1. Imports & Configuration

### Clone Repository from GitHub

Clone the KHOTAA repository to get all utility files and modules.

In [19]:
# Clone your repository from GitHub to get all utility files
import os

# Check if already cloned
if os.path.exists('/content/KHOTAA'):
    print("Repository already exists, pulling latest changes...")
    !cd /content/KHOTAA && git pull
else:
    print("Cloning repository from GitHub...")
    !cd /content && git clone https://github.com/csstudentkaum/KHOTAA.git

# Navigate to the models/classification directory
%cd /content/KHOTAA/models/classification

# Verify files are present
print("\n✓ Checking for required files:")
required_files = ['dataset_loader.py', 'dataset_preprocessing.py', 'utils/']
for file in required_files:
    exists = os.path.exists(file)
    status = "✓" if exists else "✗"
    print(f"  {status} {file}")

print("\nCurrent directory:", os.getcwd())
print("Files in current directory:", os.listdir('.')[:10])  # Show first 10 files

Repository already exists, pulling latest changes...
Already up to date.
/content/KHOTAA/models/classification

✓ Checking for required files:
  ✓ dataset_loader.py
  ✓ dataset_preprocessing.py
  ✗ utils/

Current directory: /content/KHOTAA/models/classification
Files in current directory: ['dataset_loader.py', 'dataset_preprocessing.py', 'mobilenet.ipynb', 'resnet50.ipynb', 'densenet.ipynb', '__pycache__', 'pfcnn_drnn.ipynb', 'resnet101.ipynb', 'efficientnetv2s.ipynb', 'googlenet.ipynb']
Already up to date.
/content/KHOTAA/models/classification

✓ Checking for required files:
  ✓ dataset_loader.py
  ✓ dataset_preprocessing.py
  ✗ utils/

Current directory: /content/KHOTAA/models/classification
Files in current directory: ['dataset_loader.py', 'dataset_preprocessing.py', 'mobilenet.ipynb', 'resnet50.ipynb', 'densenet.ipynb', '__pycache__', 'pfcnn_drnn.ipynb', 'resnet101.ipynb', 'efficientnetv2s.ipynb', 'googlenet.ipynb']


In [10]:
import sys
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import resnet101, ResNet101_Weights
import numpy as np
from sklearn.model_selection import StratifiedKFold
import importlib

# Add parent directory to path to import utils
parent_dir = os.path.dirname(os.getcwd())  # Go up to 'models' directory
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

# Import custom modules
from dataset_loader import SplitFolderDatasetLoader
from dataset_preprocessing import DFUPreprocessing
from utils import checkpoint_manager, training_engine
importlib.reload(checkpoint_manager)  # Reload to remove N/A print
importlib.reload(training_engine)  # Reload to get latest fixes
from utils.checkpoint_manager import CheckpointManager
from utils.training_engine import TrainingEngine, create_optimizer
from utils.metrics_evaluator import (
    calculate_metrics, print_metrics, plot_confusion_matrix,
    plot_roc_curve, plot_training_history
)

print("✓ Imports complete (modules reloaded)")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

✓ Imports complete (modules reloaded)
PyTorch version: 2.9.0+cu126
CUDA available: True


## 2. Load Dataset

### Dataset Setup - Google Drive

Mount Google Drive and configure the dataset path. Make sure you've uploaded the dataset folder to your Google Drive first.

In [ ]:
# ===== GOOGLE COLAB: MOUNT GOOGLE DRIVE =====

from google.colab import drive
import os

print("📁 Mounting Google Drive...")
drive.mount('/content/drive')

print("\n✅ Google Drive mounted successfully!")
print("\n" + "=" * 60)
print("DATASET SETUP")
print("=" * 60)

# TODO: Update this path to match YOUR Google Drive structure
# After mounting, navigate to your Drive in the Files sidebar to find the correct path
dataset_path = '/content/drive/MyDrive/dfu-dataset-annotated-into-4-classes'

print(f"\n📂 Looking for dataset at: {dataset_path}")

# Verify dataset exists
if os.path.exists(dataset_path):
    train_dir = os.path.join(dataset_path, 'train')
    if os.path.exists(train_dir):
        classes = sorted([c for c in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, c))])
        print(f"✅ Dataset found!")
        print(f"✓ Train directory exists")
        print(f"✓ Found {len(classes)} classes: {classes}")
    else:
        print(f"❌ ERROR: Train directory not found at {train_dir}")
        print("Please check your dataset structure")
else:
    print(f"❌ ERROR: Dataset not found at {dataset_path}\n")
    print("Please do ONE of the following:")
    print("\n1. UPDATE THE PATH above to match your Google Drive location")
    print("   - Look in the Files sidebar (📁) on the left")
    print("   - Navigate to: drive → MyDrive → [your folder]")
    print("   - Copy the full path")
    print("\n2. OR UPLOAD the dataset to your Google Drive:")
    print("   - Go to drive.google.com")
    print("   - Upload 'dfu-dataset-annotated-into-4-classes' folder")
    print("   - Then update the dataset_path above")
    print("\nExpected structure:")
    print("  dfu-dataset-annotated-into-4-classes/")
    print("  ├── train/")
    print("  │   ├── Grade 1/")
    print("  │   ├── Grade 2/")
    print("  │   ├── Grade 3/")
    print("  │   └── Grade 4/")
    print("  ├── valid/")
    print("  └── test/")
    
print("=" * 60)

📁 Dataset Setup for Google Colab

❌ Dataset not found!

UPLOAD DATASET TO COLAB
Follow these steps:

1. In Google Colab, look at the LEFT SIDEBAR
   - Click the 📁 Files icon

2. You'll see the /content folder
   - Right-click on /content
   - Select 'Upload'

3. Upload the entire 'dfu-dataset-annotated-into-4-classes' folder
   - Make sure it contains: train/, valid/, test/ folders
   - Each should have: Grade 1/, Grade 2/, Grade 3/, Grade 4/

4. After upload, the structure should be:
   /content/dfu-dataset-annotated-into-4-classes/
   ├── train/
   │   ├── Grade 1/
   │   ├── Grade 2/
   │   ├── Grade 3/
   │   └── Grade 4/
   ├── valid/
   └── test/

5. Re-run this cell after upload to verify

⚠️  Expected path: /content/dfu-dataset-annotated-into-4-classes
⚠️  Please upload the dataset before continuing!


In [16]:
# Load dataset using the path from the previous cell
loader = SplitFolderDatasetLoader(root_dir=dataset_path)
classes = loader.get_classes()
num_classes = loader.get_num_classes()

print(f"Classes: {classes}")
print(f"Number of classes: {num_classes}")

# Initialize preprocessing
preprocessor = DFUPreprocessing()
train_transform = preprocessor.get_train_transforms()
val_test_transform = preprocessor.get_valid_test_transforms()

# Dataset class
class DFUDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        from PIL import Image
        image = Image.open(self.image_paths[idx]).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, self.labels[idx]

# Prepare data for cross-validation
X_train, y_train = loader.load_split_paths('train', shuffle=True)
X_val, y_val = loader.load_split_paths('valid')
X_all = np.concatenate([X_train, X_val])  # Concatenate numpy arrays
y_all = np.concatenate([y_train, y_val])

# Test set (untouched until final evaluation)
X_test, y_test = loader.load_split_paths('test')
test_dataset = DFUDataset(X_test, y_test, transform=val_test_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)  # num_workers=0 to avoid multiprocessing issues

# Initialize 5-fold stratified cross-validation
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"\nTotal training samples (train+valid): {len(X_all)}")
print(f"Test samples: {len(X_test)}")
print("✓ Dataset loaded and ready for 5-fold cross-validation")

ValueError: Expected 'train' directory at: /content/KHOTAA/dfu-dataset-annotated-into-4-classes/train

## 3. Model Definition

In [ ]:
# Setup device and loss function
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
criterion = nn.CrossEntropyLoss()

print(f"Device: {device}")

# Create ResNet101 model
def create_resnet101_model(num_classes=4, pretrained=True):
    """
    Create ResNet101 model for DFU classification.
    
    Args:
        num_classes: Number of output classes (4 for DFU grades)
        pretrained: Use ImageNet pretrained weights
    
    Returns:
        ResNet101 model configured for DFU classification
    """
    if pretrained:
        model = models.resnet101(weights=ResNet101_Weights.IMAGENET1K_V2)
    else:
        model = models.resnet101(weights=None)
    
    # Modify final fully connected layer
    # ResNet fc is: Linear(2048 -> 1000)
    # Replace with: Linear(2048 -> num_classes)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    
    return model

# Test model creation
test_model = create_resnet101_model(num_classes=num_classes)
print(f"\n✓ ResNet101 model created")
print(f"Input size: 224x224")
print(f"Output classes: {num_classes}")
print(f"Final FC layer: {test_model.fc}")

## 4. Training

In [ ]:
# 5-Fold Cross-Validation Training
fold_results = []

for fold, (train_idx, val_idx) in enumerate(kfold.split(X_all, y_all), 1):
    print(f"\n{'='*60}\nFOLD {fold}/5\n{'='*60}")
    
    # Prepare fold data (X_all is numpy array of paths)
    X_train_fold = X_all[train_idx]
    y_train_fold = y_all[train_idx]
    X_val_fold = X_all[val_idx]
    y_val_fold = y_all[val_idx]
    
    train_dataset = DFUDataset(X_train_fold, y_train_fold, transform=train_transform)
    val_dataset = DFUDataset(X_val_fold, y_val_fold, transform=val_test_transform)
    
    # num_workers=0 to avoid multiprocessing issues with notebook-defined classes
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)
    
    # Create model
    model = create_resnet101_model(num_classes=num_classes, pretrained=True)
    model = model.to(device)
    
    # Setup optimizer using helper function (SGD with momentum=0.8)
    optimizer = create_optimizer(model, lr=0.001)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
    checkpoint_manager = CheckpointManager(base_dir='checkpoints', experiment_name=f'resnet101_fold{fold}')
    engine = TrainingEngine(model=model, device=device)
    
    # Train
    history = engine.train(
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        num_epochs=30,
        scheduler=scheduler,
        checkpoint_manager=checkpoint_manager,
        early_stopping_patience=7,
        use_early_stopping=True,
        verbose=True
    )
    
    # Store results
    best_val_acc = max(history['val_acc'])
    fold_results.append({
        'fold': fold,
        'best_val_acc': best_val_acc,
        'final_val_acc': history['val_acc'][-1],
        'stopped_epoch': history['stopped_epoch'],
        'history': history
    })
    print(f"Fold {fold} - Best Acc: {best_val_acc*100:.2f}% (stopped at epoch {history['stopped_epoch']})")

# Cross-validation summary
avg_acc = np.mean([r['best_val_acc'] for r in fold_results])
std_acc = np.std([r['best_val_acc'] for r in fold_results])
avg_epochs = np.mean([r['stopped_epoch'] for r in fold_results])

print(f"\n{'='*60}")
print(f"5-FOLD CROSS-VALIDATION RESULTS")
print(f"{'='*60}")
print(f"Mean Accuracy: {avg_acc*100:.2f}% ± {std_acc*100:.2f}%")
print(f"Average Epochs: {avg_epochs:.1f}")
print(f"\nIndividual Fold Results:")
for r in fold_results:
    print(f"  Fold {r['fold']}: {r['best_val_acc']*100:.2f}% (epoch {r['stopped_epoch']})")
print(f"{'='*60}")

## 5. Evaluation & Plots

In [ ]:
# Test Set Evaluation
print("\n" + "="*60)
print("TEST SET EVALUATION")
print("="*60)

# Load best fold model
best_fold_idx = np.argmax([r['best_val_acc'] for r in fold_results])
best_fold_num = fold_results[best_fold_idx]['fold']

print(f"Loading best model from Fold {best_fold_num}")

checkpoint_manager = CheckpointManager(base_dir='checkpoints', experiment_name=f'resnet101_fold{best_fold_num}')

# Load best model using the checkpoint manager
model = checkpoint_manager.load_best_model(
    fold_index=0,  # Using 0 since we create separate managers per fold
    create_model_fn=lambda: create_resnet101_model(num_classes=num_classes, pretrained=False),
    metric_name='accuracy'
)
model = model.to(device)

engine = TrainingEngine(model=model, device=device)

# Evaluate with inference time tracking
test_loss, test_acc, predictions, true_labels, inference_time = engine.evaluate(
    test_loader, 
    criterion, 
    measure_inference_time=True
)

print(f"\nTest Accuracy: {test_acc*100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")
print(f"\nInference Time Statistics:")
print(f"  Total Time: {inference_time['total_time']:.4f}s")
print(f"  Avg Time/Batch: {inference_time['avg_time_per_batch']*1000:.2f}ms ± {inference_time['std_time_per_batch']*1000:.2f}ms")
print(f"  Avg Time/Image: {inference_time['avg_time_per_image']*1000:.2f}ms")
print(f"  Throughput: {inference_time['images_per_second']:.1f} images/second")

# Get probabilities for AUC
model.eval()
all_probs = []
with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs.to(device))
        probs = torch.softmax(outputs, dim=1)
        all_probs.append(probs.cpu().numpy())

y_pred_proba = np.vstack(all_probs)

# Calculate all metrics
metrics = calculate_metrics(
    y_true=true_labels,
    y_pred=predictions,
    y_pred_proba=y_pred_proba,
    class_names=classes,
    average='macro'
)

print("\n" + "="*60)
print_metrics(metrics, title="ResNet101 Test Results")
print("="*60)

# Visualizations
import os
os.makedirs('results', exist_ok=True)

# Confusion Matrix
plot_confusion_matrix(
    y_true=true_labels,
    y_pred=predictions,
    class_names=classes,
    normalize=True,
    save_path='results/resnet101_confusion_matrix.png'
)
print("\n✓ Confusion matrix saved to results/resnet101_confusion_matrix.png")

# ROC Curve
plot_roc_curve(
    y_true=true_labels,
    y_pred_proba=y_pred_proba,
    class_names=classes,
    save_path='results/resnet101_roc_curve.png'
)
print("✓ ROC curve saved to results/resnet101_roc_curve.png")

# Training History (best fold)
plot_training_history(
    fold_results[best_fold_idx]['history'],
    save_path='results/resnet101_training_history.png'
)
print("✓ Training history saved to results/resnet101_training_history.png")

# Summary for Model Comparison
print("\n" + "="*60)
print("SUMMARY FOR MODEL COMPARISON")
print("="*60)
print(f"Model: ResNet101")
print(f"Cross-Validation Accuracy: {avg_acc*100:.2f}% ± {std_acc*100:.2f}%")
print(f"Test Accuracy: {test_acc*100:.2f}%")
print(f"Test F1-Score: {metrics['f1_score']:.4f}")
print(f"Test MCC: {metrics['mcc']:.4f}")
print(f"Test AUC: {metrics['auc']:.4f}")
print(f"Average Training Epochs: {avg_epochs:.1f}")
print(f"Inference Time: {inference_time['avg_time_per_image']*1000:.2f}ms per image")
print(f"Throughput: {inference_time['images_per_second']:.1f} images/second")
print("="*60)

## 6. Save Results for Model Comparison

Save the results for later comparison with other models (ResNet50, MobileNetV2, DenseNet, GoogLeNet, EfficientNetV2S, PFCNN+DRNN).

In [ ]:
# Save results for model comparison
import json

resnet101_results = {
    'model_name': 'ResNet101',
    'cv_results': {
        'val_accuracy': {'mean': float(avg_acc), 'std': float(std_acc)},
        'avg_epochs': float(avg_epochs),
        'fold_results': [
            {
                'fold': r['fold'],
                'best_val_acc': float(r['best_val_acc']),
                'stopped_epoch': int(r['stopped_epoch'])
            }
            for r in fold_results
        ]
    },
    'test_results': {
        'test_accuracy': float(test_acc),
        'test_loss': float(test_loss),
        'precision': float(metrics['precision']),
        'recall': float(metrics['recall']),
        'f1_score': float(metrics['f1_score']),
        'specificity': float(metrics['specificity']),
        'sensitivity': float(metrics['sensitivity']),
        'mcc': float(metrics['mcc']),
        'auc': float(metrics['auc'])
    },
    'inference_time': {
        'total_time': float(inference_time['total_time']),
        'avg_time_per_image_ms': float(inference_time['avg_time_per_image'] * 1000),
        'throughput_fps': float(inference_time['images_per_second'])
    }
}

# Save to JSON
os.makedirs('results', exist_ok=True)
with open('results/resnet101_results.json', 'w') as f:
    json.dump(resnet101_results, f, indent=4)

print("✓ Results saved to results/resnet101_results.json")
print("\nThese results can be used with the ModelComparison utility:")
print("from utils.model_comparison import ModelComparison")
print("comparison = ModelComparison()")
print("comparison.add_model_result(**resnet101_results)")
print("\n✓ ResNet101 training complete!")